# Paper 4: Word2Vec KoHA + BM25 + TF-IGM Linear SVM & CNN with Neural Seq2Seq Commentary (Namee et al., 2023)


In [ ]:
import os
import re
import math
import json
import time
import shutil
import random
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers.modeling_outputs import BaseModelOutput
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score, accuracy_score

import nltk
for pkg in ["stopwords", "wordnet"]:
    try:
        nltk.data.find(f"corpora/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 1. Cấu hình Tham số theo Mục 2.2 & 2.3 của Paper 4


In [ ]:
class Paper4Config:
    PARQUET_CANDIDATES = [
        "/kaggle/input/dap_dts/hotel_reviews.parquet",
        "/kaggle/input/dap-dts/hotel_reviews.parquet",
        "/kaggle/input/hotel-reviews-cram-absa/hotel_reviews.parquet",
        "/kaggle/input/tripadvisor-hotel-reviews/hotel_reviews.parquet",
        "data/df/hotel_reviews.parquet",
        "hotel_reviews.parquet"
    ]
    OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "model")
    LOCAL_MODEL_DIR = Path("model")
    MODEL_SAVE_PATH = OUTPUT_DIR / "modelmethod4.pt"
    LOCAL_MODEL_SAVE_PATH = LOCAL_MODEL_DIR / "modelmethod4.pt"
    HISTORY_SAVE_PATH = OUTPUT_DIR / "history_method4.json"

    GEN_MODEL_NAME = "google/flan-t5-small"
    NUM_SOFT_PROMPT_TOKENS = 4
    MAX_COMMENT_LEN = 48

    CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
    NUM_ASPECTS = 6
    NUM_SENTIMENT_CLASSES = 4
    NUM_STARS = 5

    W2V_DIM = 100
    W2V_WINDOW = 5
    W2V_MIN_COUNT = 5

    BM25_B = 0.8
    BM25_K1 = 2.0
    TFIGM_LAMBDA = 7.0
    MAX_VOCAB = 6000

    CNN_REGION_SIZES = (2, 3, 4)
    CNN_FILTERS_PER_REGION = 2
    MAX_SEQ_LEN = 128
    BATCH_SIZE = 64
    EPOCHS = 8
    LR = 1e-3

cfg = Paper4Config()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)


## 2. Tiền xử lý N-gram, Lemmatization và Hạt giống Từ khóa KoHA (Table 1 & Table 3 của Paper 4)


In [ ]:
try:
    from nltk.corpus import stopwords
    from nltk.stem import WordNetLemmatizer
    STOPWORDS_SET = set(stopwords.words("english"))
    lemmatizer = WordNetLemmatizer()
    def lemmatize_tok(w: str) -> str:
        return lemmatizer.lemmatize(w)
except Exception:
    STOPWORDS_SET = {"a", "an", "the", "and", "or", "is", "are", "was", "were", "to", "of", "in", "on", "for", "with"}
    def lemmatize_tok(w: str) -> str:
        return w

SEED_KOHA_KEYWORDS = {
    "Service": ["staff", "service", "friendly", "helpful", "quick", "efficient", "hospitality", "reception", "polite", "attentive", "manager", "concierge", "checkin", "checkout", "waiter"],
    "Facility": ["room", "facility", "facilities", "maintenance", "traffic", "clean", "cleanliness", "comfortable", "bed", "bathroom", "shower", "spacious", "modern", "quiet", "ac", "view"],
    "Experience": ["stay", "experience", "lovely", "perfect", "vacation", "holiday", "relaxing", "atmosphere", "overall", "pleasant", "memorable", "night", "trip", "enjoy", "peaceful"],
    "Amenity": ["kitchen", "food", "taste", "tasty", "breakfast", "pool", "buffet", "spa", "wifi", "restaurant", "bar", "gym", "coffee", "shuttle", "lounge", "massage"],
    "Loyalty": ["return", "back", "again", "times", "visit", "recommend", "recommended", "definitely", "repeat", "future", "choice", "staying", "next", "choose"],
    "Branding": ["price", "value", "expensive", "inexpensive", "luxury", "star", "resort", "brand", "standard", "worth", "prestigious", "class", "reputation"]
}

HEDGING_MODIFIERS = {
    "moderate", "okay", "ok", "decent", "fair", "acceptable", "average", "slightly",
    "bit", "ordinary", "normal", "passable", "tolerable", "adequate", "fine", "basic"
}

ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(cfg.CANONICAL_ASPECTS)}
SENTIMENT_TO_ID = {"Negative": 1, "Neutral": 2, "Positive": 3}
ID_TO_SENTIMENT = {0: "Not Mentioned", 1: "Negative", 2: "Neutral", 3: "Positive"}

def paper4_preprocess_tokens(text: str) -> List[str]:
    norm = unicodedata.normalize("NFC", str(text)).lower()
    unigrams = [lemmatize_tok(t) for t in re.findall(r"[a-z]+", norm) if t not in STOPWORDS_SET and len(t) > 1]
    bigrams = [f"{unigrams[i]}_{unigrams[i+1]}" for i in range(len(unigrams) - 1)]
    return unigrams + bigrams

def parse_annotations_safe(anns: Any) -> List[Dict[str, Any]]:
    if anns is None:
        return []
    if isinstance(anns, np.ndarray):
        anns = anns.tolist()
    elif isinstance(anns, str):
        try:
            anns = json.loads(anns)
        except Exception:
            return []
    if not isinstance(anns, list):
        return []
    return [dict(a) for a in anns if isinstance(a, dict)]


## 3. Nạp Dữ liệu và Huấn luyện Word2Vec Mở rộng Từ điển KoHA (Mục 2.2.1 Paper 4)


In [ ]:
parquet_path = next((Path(p) for p in cfg.PARQUET_CANDIDATES if Path(p).exists()), None)
if parquet_path is None and Path("/kaggle/input").exists():
    found = list(Path("/kaggle/input").rglob("hotel_reviews.parquet"))
    if found:
        parquet_path = found[0]
if parquet_path is None:
    raise FileNotFoundError("Không tìm thấy hotel_reviews.parquet!")

df_master = pd.read_parquet(parquet_path)
if "ordinal_rating" not in df_master.columns:
    df_master["ordinal_rating"] = df_master["rating_5"].clip(1, 5).astype(int) - 1

df_train = df_master[df_master["split"] == "train"].reset_index(drop=True)
df_val = df_master[df_master["split"] == "val"].reset_index(drop=True)
df_test = df_master[df_master["split"] == "test"].reset_index(drop=True)

train_sentences_tokens = []
for txt in df_train["review_text"]:
    for s in re.split(r"[.!?]+", str(txt)):
        toks = paper4_preprocess_tokens(s)
        if toks:
            train_sentences_tokens.append(toks)

word_freq = Counter(t for s in train_sentences_tokens for t in s)
vocab_list = [w for w, c in word_freq.most_common(cfg.MAX_VOCAB) if c >= cfg.W2V_MIN_COUNT]
word2idx = {w: i + 1 for i, w in enumerate(vocab_list)}
idx2word = {i + 1: w for i, w in enumerate(vocab_list)}
vocab_size = len(vocab_list) + 1

cooc_mat = np.zeros((vocab_size, cfg.W2V_DIM), dtype=np.float32)
rng = np.random.default_rng(SEED)
random_proj = rng.normal(0, 1.0 / math.sqrt(cfg.W2V_DIM), size=(vocab_size, cfg.W2V_DIM)).astype(np.float32)

for toks in train_sentences_tokens:
    ids = [word2idx[t] for t in toks if t in word2idx]
    for i, center_id in enumerate(ids):
        left = max(0, i - cfg.W2V_WINDOW)
        right = min(len(ids), i + cfg.W2V_WINDOW + 1)
        for j in range(left, right):
            if i != j:
                cooc_mat[center_id] += random_proj[ids[j]]

norms = np.linalg.norm(cooc_mat, axis=1, keepdims=True)
w2v_embeddings = np.divide(cooc_mat, np.maximum(norms, 1e-6))

koha_Expanded = {}
for asp, seeds in SEED_KOHA_KEYWORDS.items():
    seed_ids = [word2idx[s] for s in seeds if s in word2idx]
    if seed_ids:
        proto = w2v_embeddings[seed_ids].mean(axis=0)
        proto = proto / max(np.linalg.norm(proto), 1e-6)
        sims = w2v_embeddings @ proto
        top_ids = np.argsort(sims)[-120:][::-1]
        expanded = set(seeds) | {idx2word[i] for i in top_ids if i in idx2word and sims[i] > 0.25}
        koha_Expanded[asp] = expanded
    else:
        koha_Expanded[asp] = set(seeds)

print("KoHA Corpus size per aspect:", {k: len(v) for k, v in koha_Expanded.items()})


## 4. Thuật toán BM25 Gom cụm Câu theo Khía cạnh (Eq. 14–15) & Trọng số TF-IGM (Eq. 1–3)


In [ ]:
N_sents = max(len(train_sentences_tokens), 1)
avg_sent_len = float(np.mean([len(s) for s in train_sentences_tokens])) if train_sentences_tokens else 12.0
doc_freq_sent = Counter()
for s in train_sentences_tokens:
    for tok in set(s):
        doc_freq_sent[tok] += 1

def compute_bm25_score(query_terms: set, sent_tokens: List[str]) -> float:
    if not sent_tokens or not query_terms:
        return 0.0
    s_len = len(sent_tokens)
    tf_map = Counter(sent_tokens)
    score = 0.0
    for q in query_terms:
        if q in tf_map:
            f_qs = tf_map[q]
            df_q = doc_freq_sent.get(q, 0)
            idf_q = math.log((N_sents - df_q + 0.5) / (df_q + 0.5) + 1.0)
            num = f_qs * (cfg.BM25_K1 + 1.0)
            den = f_qs + cfg.BM25_K1 * (1.0 - cfg.BM25_B + cfg.BM25_B * (s_len / max(avg_sent_len, 1.0)))
            score += idf_q * (num / max(den, 1e-6))
    return score

def assemble_aspect_sentences_bm25(review_text: str) -> Dict[str, List[str]]:
    raw_sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", str(review_text)) if s.strip()]
    if not raw_sents and str(review_text).strip():
        raw_sents = [str(review_text).strip()]
    clusters = {asp: [] for asp in cfg.CANONICAL_ASPECTS}
    for s in raw_sents:
        s_toks = paper4_preprocess_tokens(s)
        for asp in cfg.CANONICAL_ASPECTS:
            sc = compute_bm25_score(koha_Expanded[asp], s_toks)
            if sc > 0.0:
                clusters[asp].append((sc, s))
    result = {}
    for asp in cfg.CANONICAL_ASPECTS:
        items = sorted(clusters[asp], key=lambda x: x[0], reverse=True)
        result[asp] = [s for _, s in items]
    return result

class_term_df = np.zeros((vocab_size, cfg.NUM_SENTIMENT_CLASSES), dtype=np.float32)
for _, row in df_train.iterrows():
    anns = parse_annotations_safe(row.get("aspect_annotations", []))
    for ann in anns:
        sent_cls = SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2)
        toks = set(paper4_preprocess_tokens(ann.get("span_text", "")))
        for t in toks:
            if t in word2idx:
                class_term_df[word2idx[t], sent_cls] += 1.0

igm_weights = np.zeros(vocab_size, dtype=np.float32)
for v_i in range(1, vocab_size):
    sorted_f = np.sort(class_term_df[v_i])[::-1]
    denom = sum(sorted_f[r] * (r + 1) for r in range(len(sorted_f)))
    igm_weights[v_i] = float(sorted_f[0] / denom) if denom > 0 else 0.0

def build_sparse_tfigm_matrix(texts: List[str]) -> sp.csr_matrix:
    rows, cols, vals = [], [], []
    for r_i, text_str in enumerate(texts):
        toks = paper4_preprocess_tokens(text_str)
        if not toks:
            continue
        counts = Counter(toks)
        r_cols, r_vals = [], []
        for t, freq in counts.items():
            if t in word2idx:
                vid = word2idx[t]
                w_val = math.log(1.0 + freq) * (1.0 + cfg.TFIGM_LAMBDA * igm_weights[vid])
                r_cols.append(vid)
                r_vals.append(w_val)
        if r_vals:
            norm = math.sqrt(sum(v * v for v in r_vals))
            for c_i, v_i in zip(r_cols, r_vals):
                rows.append(r_i)
                cols.append(c_i)
                vals.append(v_i / max(norm, 1e-6))
    return sp.csr_matrix((vals, (rows, cols)), shape=(len(texts), vocab_size), dtype=np.float32)

def vectorize_tf_igm(text_str: str) -> np.ndarray:
    return build_sparse_tfigm_matrix([text_str]).toarray()[0]


## 5. Huấn luyện Mô hình BM25 + TF-IGM Linear SVM, Multi-Region CNN và Neural Seq2Seq Generator


In [ ]:
gen_tokenizer = AutoTokenizer.from_pretrained(cfg.GEN_MODEL_NAME)

class Paper4MultiRegionCNN(nn.Module):
    def __init__(self, cfg: Paper4Config, pretrained_w2v: np.ndarray, svm_weights: np.ndarray, svm_biases: np.ndarray, star_svm_w: np.ndarray, star_svm_b: np.ndarray):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.max_comment_len = getattr(cfg, "MAX_COMMENT_LEN", 48)
        self.num_prompt_tokens = getattr(cfg, "NUM_SOFT_PROMPT_TOKENS", 4)

        self.embedding = nn.Embedding.from_pretrained(torch.tensor(pretrained_w2v, dtype=torch.float32), freeze=False, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=cfg.W2V_DIM, out_channels=cfg.CNN_FILTERS_PER_REGION, kernel_size=k)
            for k in cfg.CNN_REGION_SIZES
        ])
        total_filters = len(cfg.CNN_REGION_SIZES) * cfg.CNN_FILTERS_PER_REGION
        self.aspect_fc = nn.ModuleList([
            nn.Linear(total_filters, cfg.NUM_SENTIMENT_CLASSES)
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.aspect_star_fc = nn.ModuleList([
            nn.Linear(total_filters, cfg.NUM_STARS)
            for _ in range(cfg.NUM_ASPECTS)
        ])
        self.star_fc = nn.Linear(total_filters, cfg.NUM_STARS)

        self.gen_model = AutoModelForSeq2SeqLM.from_pretrained(cfg.GEN_MODEL_NAME)
        self.gen_d_model = self.gen_model.config.d_model
        for param in self.gen_model.parameters():
            param.requires_grad = False

        self.aspect_soft_prompt = nn.ModuleList([
            nn.Sequential(
                nn.Linear(total_filters, 128),
                nn.GELU(),
                nn.Linear(128, self.num_prompt_tokens * self.gen_d_model),
                nn.LayerNorm(self.num_prompt_tokens * self.gen_d_model)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.register_buffer("svm_weights", torch.tensor(svm_weights, dtype=torch.float32))
        self.register_buffer("svm_biases", torch.tensor(svm_biases, dtype=torch.float32))
        self.register_buffer("star_svm_w", torch.tensor(star_svm_w, dtype=torch.float32))
        self.register_buffer("star_svm_b", torch.tensor(star_svm_b, dtype=torch.float32))

    def forward_cnn(self, token_ids: torch.Tensor):
        bsz = token_ids.size(0)
        emb = self.embedding(token_ids).transpose(1, 2)
        pooled = []
        for conv in self.convs:
            c = F.relu(conv(emb))
            p = F.max_pool1d(c, kernel_size=c.size(-1)).squeeze(-1)
            pooled.append(p)
        feat = torch.cat(pooled, dim=-1)
        asp_logits = torch.stack([fc(feat) for fc in self.aspect_fc], dim=1)
        asp_star_logits = torch.stack([fc(feat) for fc in self.aspect_star_fc], dim=1)
        star_logits = self.star_fc(feat)
        soft_prompts = torch.stack([
            sp_head(feat).view(bsz, self.num_prompt_tokens, self.gen_d_model)
            for sp_head in self.aspect_soft_prompt
        ], dim=1)
        return asp_logits, asp_star_logits, star_logits, soft_prompts

    @torch.no_grad()
    def generate_aspect_commentary(
        self,
        aspect_idx: int,
        aspect_name: str,
        sentiment_label: str,
        star_val: int,
        focused_evidence: str,
        soft_prompts: torch.Tensor,
        seq2seq_tokenizer
    ) -> str:
        device = soft_prompts.device
        prompt_str = (
            f"Paraphrase and summarize the guest's opinion on the hotel {aspect_name} "
            f"({sentiment_label}, {star_val}/5 stars) into a short third-person analytical comment: {focused_evidence}"
        )
        p_enc = seq2seq_tokenizer(prompt_str, max_length=192, truncation=True, return_tensors="pt").to(device)
        enc_out = self.gen_model.encoder(input_ids=p_enc["input_ids"], attention_mask=p_enc["attention_mask"])
        soft_p = soft_prompts[0:1, aspect_idx, :, :]
        combined_hidden = torch.cat([0.20 * soft_p, enc_out.last_hidden_state], dim=1)
        prompt_mask = torch.ones((1, soft_p.size(1)), dtype=p_enc["attention_mask"].dtype, device=device)
        combined_mask = torch.cat([prompt_mask, p_enc["attention_mask"]], dim=1)

        gen_ids = self.gen_model.generate(
            encoder_outputs=BaseModelOutput(last_hidden_state=combined_hidden),
            attention_mask=combined_mask,
            max_new_tokens=self.max_comment_len,
            min_length=8,
            num_beams=4,
            no_repeat_ngram_size=3,
            repetition_penalty=1.25,
            early_stopping=True
        )
        return seq2seq_tokenizer.decode(gen_ids[0], skip_special_tokens=True).strip()

svm_weights_arr = np.zeros((cfg.NUM_ASPECTS, cfg.NUM_SENTIMENT_CLASSES, vocab_size), dtype=np.float32)
svm_biases_arr = np.zeros((cfg.NUM_ASPECTS, cfg.NUM_SENTIMENT_CLASSES), dtype=np.float32)

X_train_sparse = build_sparse_tfigm_matrix(df_train["review_text"].tolist())
y_train_star = df_train["ordinal_rating"].values.astype(int)

star_svm = LinearSVC(C=1.0, random_state=SEED)
star_svm.fit(X_train_sparse, y_train_star)
star_svm_w = np.zeros((cfg.NUM_STARS, vocab_size), dtype=np.float32)
star_svm_b = np.zeros(cfg.NUM_STARS, dtype=np.float32)
for idx_c, cls_v in enumerate(star_svm.classes_):
    if idx_c < star_svm.coef_.shape[0]:
        star_svm_w[int(cls_v)] = star_svm.coef_[idx_c].astype(np.float32)
        star_svm_b[int(cls_v)] = star_svm.intercept_[idx_c].astype(np.float32)

Y_train_aspects = np.zeros((len(df_train), cfg.NUM_ASPECTS), dtype=int)
Y_train_asp_stars = np.zeros((len(df_train), cfg.NUM_ASPECTS), dtype=int)
for idx_r, row in df_train.iterrows():
    ord_r = int(row["ordinal_rating"])
    Y_train_asp_stars[idx_r, :] = ord_r
    anns = parse_annotations_safe(row.get("aspect_annotations", []))
    votes = defaultdict(list)
    spans_map = defaultdict(list)
    for ann in anns:
        asp = ann.get("aspect")
        if asp in ASPECT_ID_MAP:
            a_i = ASPECT_ID_MAP[asp] - 1
            votes[a_i].append(SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2))
            spans_map[a_i].append(str(ann.get("span_text", "")))
    for a_i, v_l in votes.items():
        maj = Counter(v_l).most_common(1)[0][0]
        Y_train_aspects[idx_r, a_i] = maj
        sp_words = re.findall(r"[a-z]+", " ".join(spans_map[a_i]).lower())
        has_hedge = any(w in HEDGING_MODIFIERS for w in sp_words)
        if maj == 3:
            Y_train_asp_stars[idx_r, a_i] = 3 if has_hedge else max(3, ord_r)
        elif maj == 2:
            Y_train_asp_stars[idx_r, a_i] = 2
        else:
            Y_train_asp_stars[idx_r, a_i] = 1 if has_hedge else min(1, ord_r)

for a_i, asp in enumerate(cfg.CANONICAL_ASPECTS):
    clf_svm = LinearSVC(C=1.0, random_state=SEED)
    clf_svm.fit(X_train_sparse, Y_train_aspects[:, a_i])
    for idx_c, cls_v in enumerate(clf_svm.classes_):
        if idx_c < clf_svm.coef_.shape[0]:
            svm_weights_arr[a_i, int(cls_v)] = clf_svm.coef_[idx_c].astype(np.float32)
            svm_biases_arr[a_i, int(cls_v)] = clf_svm.intercept_[idx_c].astype(np.float32)

def encode_seq_ids(text_str: str, max_len: int = 128) -> List[int]:
    toks = paper4_preprocess_tokens(text_str)
    ids = [word2idx[t] for t in toks if t in word2idx][:max_len]
    if len(ids) < max_len:
        ids = ids + [0] * (max_len - len(ids))
    return ids

X_seq_train = torch.tensor([encode_seq_ids(t, cfg.MAX_SEQ_LEN) for t in df_train["review_text"]], dtype=torch.long)
Y_asp_train_t = torch.tensor(Y_train_aspects, dtype=torch.long)
Y_asp_star_train_t = torch.tensor(Y_train_asp_stars, dtype=torch.long)
Y_star_train_t = torch.tensor(y_train_star, dtype=torch.long)

model = Paper4MultiRegionCNN(cfg, w2v_embeddings, svm_weights_arr, svm_biases_arr, star_svm_w, star_svm_b).to(DEVICE)
with torch.no_grad():
    aspect_proto_targets = []
    for asp in cfg.CANONICAL_ASPECTS:
        kw_str = f"{asp}: " + " ".join(SEED_KOHA_KEYWORDS[asp])
        p_enc = gen_tokenizer(kw_str, return_tensors="pt", truncation=True, max_length=48).to(DEVICE)
        t5_h = model.gen_model.encoder(input_ids=p_enc["input_ids"], attention_mask=p_enc["attention_mask"]).last_hidden_state
        aspect_proto_targets.append(t5_h.mean(dim=1).squeeze(0))
    aspect_proto_targets = torch.stack(aspect_proto_targets, dim=0)

cnn_loader = DataLoader(list(zip(X_seq_train, Y_asp_train_t, Y_asp_star_train_t, Y_star_train_t)), batch_size=cfg.BATCH_SIZE, shuffle=True)
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=cfg.LR)
ce_fn = nn.CrossEntropyLoss()

for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    for bx, by_asp, by_asp_star, by_star in cnn_loader:
        bx, by_asp, by_asp_star, by_star = bx.to(DEVICE), by_asp.to(DEVICE), by_asp_star.to(DEVICE), by_star.to(DEVICE)
        optimizer.zero_grad()
        asp_l, asp_star_l, star_l, soft_p = model.forward_cnn(bx)
        pred_sp_mean = soft_p.mean(dim=2)
        target_sp = aspect_proto_targets.unsqueeze(0).expand(bx.size(0), -1, -1)
        sp_loss = (1.0 - F.cosine_similarity(pred_sp_mean, target_sp, dim=-1).mean()) + F.smooth_l1_loss(pred_sp_mean, target_sp)
        loss = (
            ce_fn(asp_l.view(-1, cfg.NUM_SENTIMENT_CLASSES), by_asp.view(-1))
            + ce_fn(asp_star_l.view(-1, cfg.NUM_STARS), by_asp_star.view(-1))
            + ce_fn(star_l, by_star)
            + 0.35 * sp_loss
        )
        loss.backward()
        optimizer.step()

def evaluate_paper4(df_eval: pd.DataFrame) -> Dict[str, Any]:
    model.eval()
    X_ev_sparse = build_sparse_tfigm_matrix(df_eval["review_text"].tolist())
    X_seq_all = torch.tensor([encode_seq_ids(t, cfg.MAX_SEQ_LEN) for t in df_eval["review_text"]], dtype=torch.long)

    y_pred_star_list = []
    Y_pred_asp_list = []
    total_loss = 0.0
    n_batches = 0

    svm_w_np = model.svm_weights.detach().cpu().numpy()
    svm_b_np = model.svm_biases.detach().cpu().numpy()
    star_w_np = model.star_svm_w.detach().cpu().numpy()
    star_b_np = model.star_svm_b.detach().cpu().numpy()

    with torch.no_grad():
        for start_i in range(0, len(df_eval), cfg.BATCH_SIZE):
            end_i = min(len(df_eval), start_i + cfg.BATCH_SIZE)
            bx_seq = X_seq_all[start_i:end_i].to(DEVICE)
            cnn_asp_l, _, cnn_star_l, _ = model.forward_cnn(bx_seq)
            cnn_asp_np = cnn_asp_l.cpu().numpy()
            cnn_star_np = cnn_star_l.cpu().numpy()

            bx_sparse = X_ev_sparse[start_i:end_i]
            svm_star_np = bx_sparse.dot(star_w_np.T) + star_b_np
            svm_asp_np = np.stack([bx_sparse.dot(svm_w_np[a].T) + svm_b_np[a] for a in range(cfg.NUM_ASPECTS)], axis=1)

            blended_star = 0.7 * svm_star_np + 0.3 * cnn_star_np
            blended_asp = 0.7 * svm_asp_np + 0.3 * cnn_asp_np

            y_pred_star_list.append(blended_star.argmax(axis=-1))
            Y_pred_asp_list.append(blended_asp.argmax(axis=-1))
            n_batches += 1

    y_pred_star = np.concatenate(y_pred_star_list, axis=0)
    Y_pred_asp = np.vstack(Y_pred_asp_list)
    y_true_star = df_eval["ordinal_rating"].values.astype(int)
    Y_true_asp = np.zeros((len(df_eval), cfg.NUM_ASPECTS), dtype=int)

    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []

    for idx_r, row in df_eval.iterrows():
        anns = parse_annotations_safe(row.get("aspect_annotations", []))
        true_toks_per_asp = defaultdict(set)
        for ann in anns:
            asp = ann.get("aspect")
            if asp in ASPECT_ID_MAP:
                a_i = ASPECT_ID_MAP[asp] - 1
                Y_true_asp[idx_r, a_i] = SENTIMENT_TO_ID.get(ann.get("sentiment", "Neutral"), 2)
                true_toks_per_asp[a_i].update(paper4_preprocess_tokens(ann.get("span_text", "")))

        bm25_clusters = assemble_aspect_sentences_bm25(row["review_text"])
        tok_aspect_count = Counter()
        for a_i, asp in enumerate(cfg.CANONICAL_ASPECTS):
            sents = bm25_clusters.get(asp, [])
            ptoks = set(paper4_preprocess_tokens(" ".join(sents[:1]))) if sents else set()
            for t in ptoks:
                tok_aspect_count[t] += 1
            t_set = true_toks_per_asp[a_i]
            rat_inter += len(ptoks & t_set)
            rat_pred_sum += len(ptoks)
            rat_true_sum += len(t_set)

        if tok_aspect_count:
            leaked = sum(1 for c in tok_aspect_count.values() if c > 1)
            overlap_rates.append(leaked / len(tok_aspect_count))

    acc0 = accuracy_score(y_true_star, y_pred_star) * 100.0
    per_star_mae = [np.abs(y_true_star[y_true_star == k] - y_pred_star[y_true_star == k]).mean() for k in range(5) if np.any(y_true_star == k)]
    macro_mae = float(np.mean(per_star_mae)) if per_star_mae else 0.0
    asp_f1s = [f1_score(Y_true_asp[:, a], Y_pred_asp[:, a], average="macro", zero_division=0) * 100.0 for a in range(cfg.NUM_ASPECTS)]
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0
    leakage_rate = float(np.mean(overlap_rates)) * 100.0 if overlap_rates else 0.0

    return {
        "val_loss": round(total_loss / max(n_batches, 1), 4),
        "acc_at_0": round(acc0, 2),
        "aspect_macro_f1": round(float(np.mean(asp_f1s)), 2),
        "macro_mae": round(macro_mae, 4),
        "rationale_f1": round(rationale_f1, 2),
        "leakage_rate": round(leakage_rate, 2),
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in asp_f1s]))
    }

val_metrics = evaluate_paper4(df_val)
test_metrics = evaluate_paper4(df_test)
print("Paper 4 Val Metrics:", val_metrics)
print("Paper 4 Test Metrics:", test_metrics)

torch.save({
    "epoch": cfg.EPOCHS,
    "method": "Paper 4: Word2Vec KoHA + BM25 + TF-IGM SVM & CNN (Namee et al., 2023)",
    "architecture": "paper4_bm25_tfigm_svm_cnn",
    "gen_model_name": cfg.GEN_MODEL_NAME,
    "model_state_dict": model.state_dict(),
    "w2v_embeddings": w2v_embeddings,
    "word2idx": word2idx,
    "igm_weights": igm_weights,
    "N_sents": N_sents,
    "avg_sent_len": avg_sent_len,
    "doc_freq_sent": dict(doc_freq_sent),
    "koha_expanded": {k: list(v) for k, v in koha_Expanded.items()},
    "canonical_aspects": cfg.CANONICAL_ASPECTS,
    "val_metrics": val_metrics,
    "test_metrics": test_metrics
}, cfg.MODEL_SAVE_PATH)
if cfg.LOCAL_MODEL_SAVE_PATH.resolve() != cfg.MODEL_SAVE_PATH.resolve():
    shutil.copy2(cfg.MODEL_SAVE_PATH, cfg.LOCAL_MODEL_SAVE_PATH)


## 6. Suy luận trên Review Mới bằng BM25 + TF-IGM SVM & CNN và Neural Seq2Seq Generator


In [ ]:
@torch.no_grad()
def predict_new_review_paper4(new_review_text: str, model: Paper4MultiRegionCNN, seq2seq_tokenizer) -> str:
    model.eval()
    raw_text = unicodedata.normalize("NFC", str(new_review_text).strip())
    clusters = assemble_aspect_sentences_bm25(raw_text)
    v_igm_global = torch.tensor(vectorize_tf_igm(raw_text), dtype=torch.float32, device=DEVICE).unsqueeze(0)
    seq_global = torch.tensor([encode_seq_ids(raw_text, cfg.MAX_SEQ_LEN)], dtype=torch.long, device=DEVICE)
    cnn_asp_g, cnn_asp_star_g, _, soft_p_g = model.forward_cnn(seq_global)
    svm_asp_g = torch.einsum("bd,acd->bac", v_igm_global, model.svm_weights) + model.svm_biases
    blended_asp_g = (0.7 * svm_asp_g + 0.3 * cnn_asp_g).squeeze(0).cpu().numpy()

    mentioned = []
    per_asp_str = {}

    for a_idx, asp in enumerate(cfg.CANONICAL_ASPECTS):
        sents = clusters.get(asp, [])
        pred_cls_g = int(np.argmax(blended_asp_g[a_idx]))
        if sents or pred_cls_g != 0:
            mentioned.append(asp)
            joined_s = " ".join(sents[:2]) if sents else raw_text
            v_igm = torch.tensor(vectorize_tf_igm(joined_s), dtype=torch.float32, device=DEVICE).unsqueeze(0)
            seq_t = torch.tensor([encode_seq_ids(joined_s, cfg.MAX_SEQ_LEN)], dtype=torch.long, device=DEVICE)

            cnn_asp_l, cnn_asp_star_l, cnn_star_l, soft_p = model.forward_cnn(seq_t)
            svm_asp_l = torch.einsum("bd,cd->bc", v_igm, model.svm_weights[a_idx]) + model.svm_biases[a_idx]
            svm_star_l = v_igm @ model.star_svm_w.t() + model.star_svm_b

            asp_probs = 0.7 * svm_asp_l[0] + 0.3 * cnn_asp_l[0, a_idx]
            pol_id = int(asp_probs[1:].argmax().item()) + 1
            sent_lbl = ID_TO_SENTIMENT[pol_id]
            star_v = int((0.5 * svm_star_l[0] + 0.5 * cnn_asp_star_l[0, a_idx]).argmax().item()) + 1
            gen_comment = model.generate_aspect_commentary(
                aspect_idx=a_idx,
                aspect_name=asp,
                sentiment_label=sent_lbl,
                star_val=star_v,
                focused_evidence=joined_s,
                soft_prompts=soft_p,
                seq2seq_tokenizer=seq2seq_tokenizer
            )
            per_asp_str[asp] = f"{star_v}/5 ★ ({sent_lbl}) - {gen_comment}"
        else:
            per_asp_str[asp] = ""

    lines = [f"- Aspect: {', '.join(mentioned) if mentioned else 'None'}"]
    for asp in cfg.CANONICAL_ASPECTS:
        lines.append(f"- {asp}: {per_asp_str[asp]}" if per_asp_str[asp] else f"- {asp}:")
    return "\n".join(lines)
